In [1]:
import dashscope
from http import HTTPStatus
import time
import os
from typing import List
from dotenv import load_dotenv
import json

load_dotenv()

# llm
def qwen_chat(messages):
    api_key = os.getenv("LLM_API_KEY") 
    if not api_key:
        raise ValueError("LLM_API_KEY not found in environment variables.")
        
    response = dashscope.Generation.call(
        api_key=api_key,
        model="qwen-plus", 
        messages=messages,
        result_format="message",
    )
    if response.status_code == HTTPStatus.OK:
        return response["output"]["choices"][0]["message"]["content"]
    else:
        print(
            f"Qwen API Error: Status code: {response.status_code}, "
            f"error code: {response.code}, error message: {response.message}"
        )
        return None

# retry
def call_api_with_retry(messages, max_retries=3, initial_delay=1):
    for attempt in range(max_retries):
        response = qwen_chat(messages)
        if response:
            return response
        
        if attempt < max_retries - 1:
            wait_time = initial_delay * (2 ** attempt)
            print(f"Retrying after {wait_time} sec...")
            time.sleep(wait_time)
            
    return None

# product_data.json
def load_products_from_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data["stores"]

# filter products
def filter_products_by_store_and_type(stores, store_name, store_quantity, item_type, allergy_list, inventory_priority):
    store = next((s for s in stores if s.get("store_name") == store_name), None)
    if not store:
        raise Exception(f"未找到商店：{store_name}")

    filtered = [p for p in store.get("products", []) if p.get("type") == item_type]

    if allergy_list:
        filtered = [
            p for p in filtered
            if not any(a in p.get("allergens", []) for a in allergy_list)
        ]

    reverse = inventory_priority == "库存最高"
    filtered.sort(key=lambda x: x.get("stock", 0), reverse=reverse)

    return filtered


def load_script_template(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def get_script_name(script_json_path: str) -> str:
    with open(script_json_path, "r", encoding="utf-8") as f:
        script = json.load(f)
    return script.get("system_config", {}).get("script_metadata", {}).get("name", "未知剧本")


def generate_full_dynamic_script(
    user_id: str,
    allergy_list: List[str],
    product_json_path: str,
    script_json_path: str,
    style: str
):
    # load products
    stores = load_products_from_json(product_json_path)

    # load script template
    script = load_script_template(script_json_path)
    tasks = script["tasks"]

    # background description
    final_script = {
        "script_metadata": script["system_config"]["script_metadata"],
        "mechanism": script.get("four_deities_collection", {}).get("mechanism", ""),
        "tasks": []
    }

    for task in tasks:
        slot = task.get("dynamic_data_slot")
        selected_item = None
        npc_dialogue = None
        sub_tasks = task.get("sub_tasks", [])

        if slot:
            try:
                item_list = filter_products_by_store_and_type(
                    stores=stores,
                    store_name=slot.get("store_name"),
                    store_quantity=slot.get("store_quantity"),
                    item_type=slot.get("item_type"),
                    allergy_list=allergy_list if slot["selection_criteria"].get("allergy_constraint") != "无" else [],
                    inventory_priority=slot["selection_criteria"].get("inventory_priority", "库存最高")
                )

                if not item_list:
                    raise Exception(f"没有可用商品满足 SLOT {slot['slot_key']} 的条件")

                selected_item = item_list[0]

                npc_dialogue = generate_npc_dialogue(
                    npc_template=slot["npc_template"],
                    item_name=selected_item["name"],
                    npc_role=task.get("npc_role", ""),
                    allergies_list=allergy_list,
                    task_name=task["stage_name"],
                    script_name=script["system_config"]["script_metadata"].get("title", "未知剧本"),
                    style=style
                )
            except KeyError as e:
                print(f"动态数据槽处理失败: {e}")
            except Exception as e:
                print(f"槽位筛选异常: {e}")

        selected_item_field = None
        if selected_item:
            product_name = selected_item.get("name")
            if product_name:
                selected_item_field = f"{product_name}"
            else:
                selected_item_field = f"{product_name}"

        final_script["tasks"].append({
            "task_id": task["task_id"],
            "stage_name": task["stage_name"],
            "location": task.get("location"),
            "npc": task.get("npc"),
            "npc_role": task.get("npc_role"),
            "objective_template": task.get("objective_template"),
            "task_type": task.get("task_type"),
            "completion_mechanism": task.get("completion_mechanism"),
            "virtual_reward": task.get("virtual_reward"),
            "selected_item": selected_item_field,
            "npc_dialogue": npc_dialogue,
            "next_task_id": task.get("next_task_id"),
            "sub_tasks": sub_tasks
        })

    return final_script


def generate_npc_dialogue(npc_template: str, item_name: str, npc_role: str, allergies_list: List[str], task_name: str, script_name: str, style: str) -> str:
    system_prompt = (
    f"你是一个高级剧本杀内容生成AI，专为《{script_name}》剧本创作NPC台词，风格为{style}。你是该NPC的扮演者。"
    f"你的任务是根据提供的[话术模板]、[动态商品]和[用户过敏信息]，生成一段符合场景、语境以及风格的NPC台词。注意话术模板是npc要完成的任务"
    f"**生成要求**：1. 必须使用提供的[动态商品]名称，但是给他取一个符合剧情道具的名字，输出为：道具名（商品名）。"
    f"2. 涉及食品时，务必巧妙地在台词中体现已排除了用户过敏原，但是纪念品类型不需要在台词中说明过敏原。"
    f"3. 只输出最终的NPC台词，不要输出任何解释性文字或多余的标点符号。"
)
    
    user_prompt = f"""
    请根据以下信息生成最终的NPC台词：    
    【任务名称】：{task_name}
    【NPC角色】：{npc_role}
    【动态商品名称】：{item_name}
    【用户主要过敏原】：{', '.join(allergies_list) if allergies_list else '无'}
    【话术模板】：{npc_template}
    """
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    
    generated_content = call_api_with_retry(messages)
    
    if generated_content:
        return generated_content.strip()
    else:
        print(f"LLM调用失败，为 {item_name} 使用回退纯文本替换。")
        dialogue = npc_template.replace("{{动态商品名称}}", item_name)
        if allergies_list:
            dialogue = dialogue.replace("{{用户主要过敏原}}", allergies_list[0])
        return dialogue.strip()


dynamic_script = generate_full_dynamic_script(
    user_id="USER_1234",
    allergy_list=["蛋", "花生", "山楂", "大米"],
    product_json_path="/home/zhangbi/Zhangbi_Traveler/DataBase/Search_Update_Context/json/pgvector/src/llm/prompts/product_data.json",
    script_json_path="/home/zhangbi/Zhangbi_Traveler/DataBase/Search_Update_Context/json/pgvector/src/llm/prompts/scripts_template.json",
    style="现代幽默"
)

output_path = "/home/zhangbi/Zhangbi_Traveler/DataBase/Search_Update_Context/json/pgvector/src/llm/prompts/3.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(dynamic_script, f, ensure_ascii=False, indent=2)

print("已保存到：", output_path)

槽位筛选异常: 未找到商店：醋坊
槽位筛选异常: 未找到商店：研学院
已保存到： /home/zhangbi/Zhangbi_Traveler/DataBase/Search_Update_Context/json/pgvector/src/llm/prompts/3.json


In [43]:
def load_task_from_json(script_json_path: str, task_id: str = None):
    """
    从 JSON 文件中加载指定任务或背景信息。
    如果 task_id 为 None，则仅返回背景信息。
    """
    with open(script_json_path, "r", encoding="utf-8") as f:
        full_script = json.load(f)
    
    if task_id is None:
        return {
            "era_background": full_script.get("script_metadata", {}).get("era_background", "未知背景"),
            "mechanism": full_script.get("mechanism", "")
        }
    
    task = next((t for t in full_script["tasks"] if t["task_id"] == task_id), None)
    if task is None:
        raise Exception(f"未找到 task_id={task_id} 的任务")
    
    return {
        "task": task,
        "era_background": full_script.get("script_metadata", {}).get("era_background", "未知背景"),
        "mechanism": full_script.get("mechanism", "")
    }

def introduce_background(script_json_path: str):
    """
    介绍剧本的背景信息。
    """
    background_data = load_task_from_json(script_json_path)
    era_background = background_data["era_background"]
    mechanism = background_data["mechanism"]
    
    introduction = f"""
    欢迎来到剧本世界！
    
    【剧本背景】
    {era_background if era_background else '暂无背景信息。'}
    
    【机制说明】
    {mechanism if mechanism else '暂无机制说明。'}
    """
    
    return introduction

def run_task_interaction(
    user_id: str,
    user_input: str,
    script_json_path: str,
    current_task_id: str,
    call_llm_func=None,
    is_first_interaction: bool = False,
    conversation_history: list = None
):
    """
    根据用户输入与剧本模板，让 LLM 生成 NPC 的回应。
    确保对话历史的独立性，仅处理当前任务的内容。
    """
    if conversation_history is None:
        conversation_history = []
    
    # 过滤出当前任务的对话历史
    task_conversation_history = [turn for turn in conversation_history if turn.get("task_id") == current_task_id]
    
    if is_first_interaction:
        background_intro = introduce_background(script_json_path)
        print(background_intro)
    
    # 加载当前任务
    task_data = load_task_from_json(script_json_path, current_task_id)
    task = task_data["task"]
    era_background = task_data["era_background"]
    mechanism = task_data["mechanism"]
    
    npc = task.get("npc", "NPC")
    npc_role = task.get("npc_role", "")
    npc_dialogue = task.get("npc_dialogue", "")
    scene = task.get("stage_name", "")
    location = task.get("location", "")
    selected_item = task.get("selected_item", None)
    objective_template = task.get("objective_template", "")
    
    # 生成摘要并加入上下文
    summary = generate_summary(task_conversation_history, current_task_id)
    
    prompt = f"""
            你现在是一个剧情 NPC，与用户进行剧本杀式互动。
            
            【背景信息】
            - 剧本背景：{era_background}
            - 机制说明：{mechanism}
            
            【对话摘要】
            {summary}
            
            【场景信息】
            - 任务：{scene}
            - 场景地点：{location}
            - 任务目标：{objective_template}
            - NPC：{npc}（{npc_role}）
            - NPC 手上相关道具：{selected_item}
            - NPC 预设台词：{npc_dialogue}
            
            【你要做的事】
            根据用户的输入，对他作出自然、沉浸式的剧情回应，要用户能听懂，引导他继续任务推进。如果有NPC预设台词则按照预设台词风格进行回应。如果没有NPC，则自己扮演一个合适的NPC角色进行回应。
            
            【用户输入】
            {user_input}
            
            请生成 NPC 的回复，不要解释，不要输出系统说明，只要角色对话。
            """
    
    messages = [
        {"role": "user", "content": prompt}
    ]
    
    npc_reply = call_llm_func(messages)
    
    # 更新对话历史
    conversation_history.append({
        "task_id": current_task_id,
        "user_input": user_input,
        "npc_response": npc_reply
    })
    
    # 返回下一步任务 ID
    next_task = task.get("next_task_id", "END")
    
    return {
        "task_id": current_task_id,
        "npc_response": npc_reply,
        "next_task_id": next_task,
        "conversation_history": conversation_history
    }

run_task_interaction(
    user_id="USER_1234",
    user_input="有什么谜题？",
    script_json_path="/home/zhangbi/Zhangbi_Traveler/DataBase/Search_Update_Context/json/pgvector/src/llm/prompts/2.json",
    current_task_id="T03_TUNNEL_SEEK",
    call_llm_func=call_api_with_retry,
    is_first_interaction=False,
    conversation_history=conversation_history
)

{'task_id': 'T03_TUNNEL_SEEK',
 'npc_response': '快看那焚香阁旁的石碑——上面刻着四句诗，字迹被岁月磨得模糊，但依稀能辨：\n\n“朱雀南向啼晨曦，  \n三足振翅不离井。  \n七宿连珠火未熄，  \n衔书当落槐阴里。”\n\n这……是朱雀位的星宿谜语！你看那井口边的老槐树，守了二十年，我从未见它投下完整的影子。更鼓将响，怕是巡夜人就来了……你说，这“衔书”是指什么？图在槐阴之下？还是……另有玄机？',
 'next_task_id': 'T04_TUNNEL_CROSS',
 'conversation_history': [{'task_id': 'T01_PROLOGUE',
   'user_input': '开始游戏！！！',
   'npc_response': '*夜色中，南堡门的灯笼微微摇曳，一个身披灰袍的身影从暗处缓步走出，目光沉静地望向你*\n\n嘘——莫要声张。今夜能在此相遇，想必是天意。我乃守门人老槐，掌管入城凭证。\n\n*抬手轻指城门上方隐约发光的朱雀纹样*\n\n你可知这四象之力？青龙、白虎、朱雀、玄武...各自镇守一方。而你面前的，正是朱雀所在之南门。唯有通过考验者，方得进入。\n\n*低声道*\n\n若真想踏入此门，便随我来吧。但记住——心诚则灵，妄念者终将迷途。准备好了吗？'},
  {'task_id': 'T01_PROLOGUE',
   'user_input': '开始游戏！！！',
   'npc_response': '*夜色中，南堡门的灯笼微微摇曳，一个身披灰袍的身影从暗处缓步走出，目光沉静地望向你*\n\n嘘——莫要声张。今夜能在此相遇，想必是天意。我乃守门人老槐，掌管入城凭证。\n\n*抬手轻指城门上方隐约发光的朱雀纹样*\n\n你可知这四象之力？青龙、白虎、朱雀、玄武...各自镇守一方。而你面前的，正是朱雀所在之南门。唯有通过考验者，方得进入。\n\n*低声道*\n\n若真想踏入此门，便随我来吧。但记住——心诚则灵，妄念者终将迷途。准备好了吗？'},
  {'task_id': 'T01_PROLOGUE',
   'user_input': '我找到了那个神秘的隧道，接下来该怎么办？',
   'npc_response': '*守门人老槐的面容在灯笼下微微